# OpenVLA-7B LoRA — MMD-SELECTIVE — LIBERO-Object (Kaggle T4)
### layer-selective-peft-vla project — Phase 2

Built directly on the proven, working baseline notebook (`openvla-baseline-uniform-lora-run.ipynb`) — sections 1–6 below are reused byte-for-byte unchanged. The only new content is: scoring each of OpenVLA's 32 transformer layers by MMD (Maximum Mean Discrepancy) between LIBERO-Spatial (source) and LIBERO-Object (target) activations, selecting the top layers to match the baseline's real trainable-parameter budget (110,828,288, confirmed from the baseline's own successful run), and training with LoRA applied ONLY to those selected layers instead of all of them.

**This notebook trains for 100 steps** — a smoke test for the selective-LoRA mechanism, mirroring the role the original 10-step smoke test played before the baseline was scaled to a real 2000-step run. Once this passes, scale MAX_STEPS the same way the baseline was scaled.

**Three specific problems in an earlier draft of this notebook, fixed here:**
1. The fp16 patch had regressed to only replacing the *first* occurrence of `torch.bfloat16` — reverted to patching every occurrence (as in the proven baseline).
2. The DDP fix had been replaced with an unverified, different mechanism (a blanket `vla.to(device_id)` with no `_set_static_graph()` call) that risked re-triggering the exact `.to()`-on-quantized-model crash this project already spent multiple rounds fixing. Reverted to the exact proven fix from the baseline.
3. The `target_modules` regex for the selected layers was never actually verified against a real instantiated model before being written into the training script — the single highest-risk gap, since a wrong regex would silently corrupt the whole experiment while still running without error. **Section 8 below adds that verification directly**, using the same `init_empty_weights` technique already proven earlier in this project (no real weights or GPU needed for this check).

## 1. Install pinned dependencies
(Unchanged from the proven baseline notebook.)

In [ ]:
import subprocess
import sys

def install(packages, no_deps=False):
    cmd = [sys.executable, "-m", "pip", "install"]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(packages)
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)

# OpenVLA compatibility pins
install([
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "peft==0.11.1",
    "accelerate>=0.25.0",
    "sentencepiece==0.1.99",
    "draccus==0.8.0",
    "json-numpy",
    "protobuf",
])

# Standard utilities
install([
    "einops",
    "jsonlines",
    "rich",
    "matplotlib",
    "huggingface_hub",
    "wandb",
])

# Newer bitsandbytes compatible with Kaggle's modern PyTorch/Triton environment
install([
    "bitsandbytes>=0.45.0",
])

# TensorFlow Datasets (we do NOT install a pinned tensorflow — Kaggle's native TF stays)
install([
    "tensorflow_datasets==4.9.3",
])

# DLIMP dataloader — not a PyPI package, installed from OpenVLA's fork.
# --no-deps prevents it from disturbing Kaggle's existing TensorFlow/PyTorch stack.
install([
    "git+https://github.com/moojink/dlimp_openvla"
], no_deps=True)

print("\n✅ All installations completed successfully!")


## 2. Configure paths and install OpenVLA (editable, no-deps)
(Unchanged.)

In [ ]:
import os
import subprocess
import sys

WORKDIR = "/kaggle/working"
OPENVLA_DIR = os.path.join(WORKDIR, "openvla")
DATA_DIR = os.path.join(WORKDIR, "datasets", "openvla", "modified_libero_rlds")
CHECKPOINT_DIR = os.path.join(WORKDIR, "openvla_checkpoints", "smoke_test_run")

if not os.path.exists(OPENVLA_DIR):
    print("Cloning OpenVLA...")
    subprocess.run(["git", "clone", "https://github.com/openvla/openvla", OPENVLA_DIR], check=True)
else:
    print("✅ OpenVLA already cloned.")

# Install OpenVLA natively into the Python environment (editable, no-deps —
# dependencies are already handled explicitly in the cell above)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", OPENVLA_DIR, "--no-deps"], check=True)
print("✅ Installed OpenVLA package.")


## 3. Patch `droid_utils.py` — remove the `tensorflow_graphics` import
(Unchanged.)

In [ ]:
patch_file = os.path.join(
    OPENVLA_DIR, "prismatic", "vla", "datasets", "rlds", "oxe", "utils", "droid_utils.py"
)
with open(patch_file, "r") as f:
    content = f.read()

target = "import tensorflow_graphics.geometry.transformation as tfg"
if target in content:
    content = content.replace(target, f"# {target}")
    with open(patch_file, "w") as f:
        f.write(content)
    print("\u2705 Patched droid_utils.py to remove tensorflow_graphics dependency.")
else:
    print("\u2139\ufe0f Exact import line not found verbatim \u2014 checking current state below.")

with open(patch_file, "r") as f:
    verify = f.read()

# Check for a LIVE (uncommented) import specifically \u2014 a naive substring check
# for "import tensorflow_graphics" fails even on a correctly-commented line, since
# "# import tensorflow_graphics..." still contains that substring.
live_imports = [
    line for line in verify.splitlines()
    if line.strip().startswith("import tensorflow_graphics")
]
assert not live_imports, (
    f"\u274c droid_utils.py still has a live (uncommented) tensorflow_graphics import: "
    f"{live_imports}. The patch did not apply \u2014 inspect the file manually before proceeding."
)
print("\u2705 Verified: no live tensorflow_graphics import remains in droid_utils.py.")


## 4. Patch `finetune.py` — use fp16 instead of bf16 for T4 compatibility
(Unchanged — patches EVERY occurrence of `torch.bfloat16`, not just the first.)

In [ ]:
finetune_file = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune.py")
with open(finetune_file, "r") as f:
    ft_content = f.read()

n_torch_dtype = ft_content.count("torch_dtype=torch.bfloat16")
n_bnb = ft_content.count("bnb_4bit_compute_dtype=torch.bfloat16")
n_dtype = ft_content.count("dtype=torch.bfloat16")
n_to = ft_content.count(".to(torch.bfloat16)")
print(f"Occurrences before patch — torch_dtype: {n_torch_dtype}, bnb_4bit_compute_dtype: {n_bnb}, "
      f"dtype=: {n_dtype}, .to(): {n_to}")
if n_torch_dtype > 1:
    print("⚠️ Multiple torch_dtype=torch.bfloat16 occurrences found — patching ALL of them, not just the first.")

# Patch every occurrence, not just the first
ft_content = ft_content.replace("bnb_4bit_compute_dtype=torch.bfloat16", "bnb_4bit_compute_dtype=torch.float16")
ft_content = ft_content.replace("torch_dtype=torch.bfloat16", "torch_dtype=torch.float16")
ft_content = ft_content.replace("dtype=torch.bfloat16", "dtype=torch.float16")
ft_content = ft_content.replace(".to(torch.bfloat16)", ".to(torch.float16)")

with open(finetune_file, "w") as f:
    f.write(ft_content)

assert "torch.bfloat16" not in ft_content, (
    "❌ A bfloat16 reference survived the patch — inspect finetune.py manually for a pattern "
    "this cell did not account for."
)
print("✅ Verified: no remaining bfloat16 references in finetune.py.")


## 5. Preflight import + version + hardware audit
(Unchanged.)

In [ ]:
import sys
from pathlib import Path
import torch
import transformers
import peft
import timm
import tensorflow as tf
import tensorflow_datasets as tfds
import bitsandbytes as bnb
import accelerate
from dlimp import DLataset

OPENVLA_DIR_PATH = Path(OPENVLA_DIR)
if str(OPENVLA_DIR_PATH) not in sys.path:
    sys.path.insert(0, str(OPENVLA_DIR_PATH))

prismatic_dir = OPENVLA_DIR_PATH / "prismatic"
print("OpenVLA directory:", OPENVLA_DIR_PATH)
print("OpenVLA exists:", OPENVLA_DIR_PATH.exists())
print("Prismatic directory:", prismatic_dir)
print("Prismatic exists:", prismatic_dir.exists())

if not prismatic_dir.exists():
    raise RuntimeError("❌ The OpenVLA repository does not contain the expected 'prismatic/' directory.")

# Import the specific module we patched, to confirm it imports cleanly post-patch
from prismatic.vla.datasets.rlds.oxe.utils import droid_utils

print("\n--- VERSION AUDIT ---")
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"timm:         {timm.__version__}")
print(f"BitsAndBytes: {bnb.__version__}")
print(f"Accelerate:  {accelerate.__version__}")
print(f"TensorFlow:   {tf.__version__}")
print(f"TFDS:         {tfds.__version__}")

print("\n--- COMPATIBILITY CHECKS ---")
print("✅ DLIMP dataset structures imported successfully.")
print("✅ Prismatic/OpenVLA source imported successfully (post-patch).")

print("\n--- HARDWARE AUDIT ---")
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"GPUs Detected: {gpu_count}")
    for i in range(gpu_count):
        props = torch.cuda.get_device_properties(i)
        print(f" - GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1024**3:.2f} GB)")
else:
    raise RuntimeError("❌ No GPU found! Enable a GPU accelerator in Kaggle notebook settings.")

print("\n✅ VERSION / COMPATIBILITY / HARDWARE AUDIT PASSED.")


## 6. Download the LIBERO-Object AND LIBERO-Spatial RLDS datasets

Baseline only downloaded LIBERO-Object (the target). This notebook also needs LIBERO-Spatial as the source dataset for MMD scoring — same download mechanism, same HF dataset repo (`openvla/modified_libero_rlds`), just a second subset.

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

print("Downloading LIBERO-Object dataset subset (target)...")
subprocess.run([
    "huggingface-cli", "download", "openvla/modified_libero_rlds",
    "--repo-type", "dataset",
    "--include", "libero_object_no_noops/*",
    "--local-dir", DATA_DIR
], check=True)

print("Downloading LIBERO-Spatial dataset subset (source for MMD comparison)...")
subprocess.run([
    "huggingface-cli", "download", "openvla/modified_libero_rlds",
    "--repo-type", "dataset",
    "--include", "libero_spatial_no_noops/*",
    "--local-dir", DATA_DIR
], check=True)

for name in ["libero_object_no_noops", "libero_spatial_no_noops"]:
    if os.path.exists(os.path.join(DATA_DIR, name)):
        print(f"✅ {name} verified!")
    else:
        raise RuntimeError(f"❌ {name} download failed or path is incorrect!")


## 7. Patch DDP + Accelerate

**Identical to the proven baseline notebook** — an earlier draft of this MMD notebook replaced this with a different, unverified mechanism. Reverted here to the exact fix that's actually been confirmed working on real Kaggle hardware.

In [ ]:
import os
import re
import shutil
import sys
import subprocess
import accelerate

finetune_file = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune.py")

# ------------------------------------------------------------------
# Fix 1: move any CPU-resident LoRA params/buffers to GPU before DDP,
# and set static graph. Applied ONCE (the smoke test applied this twice
# by accident when two cells matched the same target string in sequence;
# harmless there since the second pass found nothing left to move, but
# cleaner to do it once here).
# ------------------------------------------------------------------
with open(finetune_file, "r") as f:
    ft_content = f.read()

target_ddp = "vla = DDP(vla, device_ids=[device_id], find_unused_parameters=True, gradient_as_bucket_view=True)"

device_move_code = """
    # Manually move any CPU-resident LoRA weights/buffers to the GPU before DDP.
    # Needed because PEFT's newly-attached LoRA adapter params are created on CPU
    # by default, while the 4-bit quantized base model already lives on GPU — and
    # DDP's NCCL backend cannot broadcast CPU tensors during state sync.
    for param in vla.parameters():
        if param.device.type == 'cpu':
            param.data = param.data.to(device_id)
    for buffer in vla.buffers():
        if buffer.device.type == 'cpu':
            buffer.data = buffer.data.to(device_id)

    vla = DDP(vla, device_ids=[device_id], find_unused_parameters=True, gradient_as_bucket_view=True)
    vla._set_static_graph()  # avoids a related "double-ready" DDP crash
"""

if device_move_code.strip() in ft_content:
    print("✅ finetune.py already patched for CPU->GPU param move + static graph.")
elif target_ddp in ft_content:
    ft_content = ft_content.replace(target_ddp, device_move_code)
    with open(finetune_file, "w") as f:
        f.write(ft_content)
    print("✅ Patched finetune.py: CPU-resident params moved to GPU before DDP, static graph set.")
else:
    raise RuntimeError(
        "Could not find the expected DDP initialization line in finetune.py. "
        "This patch is CONFIRMED necessary (a real Kaggle run failed here with "
        "'RuntimeError: No backend type associated with device type cpu') \u2014 "
        "inspect finetune.py's finetune() function manually for the current DDP(...) call."
    )

# ------------------------------------------------------------------
# Fix 2: patch accelerate's dispatch_model() so it doesn't call .to()
# on an already-quantized model.
# ------------------------------------------------------------------
accelerate_dir = os.path.dirname(accelerate.__file__)
accelerate_file = os.path.join(accelerate_dir, "big_modeling.py")
print(f"Accelerate big_modeling.py: {accelerate_file}")

if not os.path.exists(accelerate_file):
    raise FileNotFoundError(f"Could not find Accelerate big_modeling.py at:\n{accelerate_file}")

with open(accelerate_file, "r") as f:
    source = f.read()

# Confirmed via real Kaggle traceback:
#   transformers/modeling_utils.py -> from_pretrained -> dispatch_model(...)
#   -> accelerate/big_modeling.py: "model.to(device)"
#   -> transformers raises: "ValueError: `.to` is not supported for `4-bit` or
#      `8-bit` bitsandbytes models."
# is_loaded_in_4bit / is_loaded_in_8bit are set by hf_quantizer.postprocess_model(),
# which runs AFTER dispatch_model() — so checking those here never blocks the call.
# model.is_quantized is set BEFORE dispatch_model() (via hf_quantizer.preprocess_model())
# and is the reliable signal, confirmed by reading transformers 4.40.1's actual source.
old_snippet = "        if device != \"disk\":\n            model.to(device)\n"
new_snippet = (
    "        if device != \"disk\":\n"
    "            if not getattr(model, \"is_quantized\", False):\n"
    "                model.to(device)\n"
)

patched = False
if new_snippet in source:
    print("✅ Accelerate is already patched.")
    patched = True
elif old_snippet in source:
    source = source.replace(old_snippet, new_snippet, 1)
    with open(accelerate_file, "w") as f:
        f.write(source)
    print("✅ Patched Accelerate's dispatch_model() single-device branch (exact match).")
    patched = True
else:
    pattern = re.compile(r'(\n(\s*)if device != "disk":\n)(\s*)model\.to\(device\)\n')
    def _sub(m):
        line_indent, body_indent = m.group(2), m.group(3)
        return (
            f'{m.group(1)}{body_indent}if not getattr(model, "is_quantized", False):\n'
            f'{body_indent}    model.to(device)\n'
        )
    new_source, n = pattern.subn(_sub, source, count=1)
    if n == 1:
        with open(accelerate_file, "w") as f:
            f.write(new_source)
        print("✅ Patched Accelerate's dispatch_model() single-device branch (regex fallback).")
        patched = True

if not patched:
    raise RuntimeError(
        "Could not patch Accelerate's dispatch_model() to skip .to() on quantized models "
        f"(installed Accelerate: {accelerate.__version__}, file: {accelerate_file}). "
        "This IS confirmed necessary \u2014 inspect big_modeling.py's dispatch_model() manually."
    )

print("\n✅ All patches applied. Ready to launch training.")


## 8. Compute MMD scores per layer, on real hardware

Loads the base model (4-bit, same as training) in eval mode, registers a forward hook on each of the 32 LLM decoder layers, runs a modest sample of LIBERO-Spatial (source) and LIBERO-Object (target) examples through it, mean-pools each layer's hidden states per example, and computes RBF-kernel MMD between the two sets of pooled activations for each layer. Any batch that fails to process is reported loudly (exception type + message, not swallowed), and every layer is asserted to have collected at least one real activation from each dataset before any score is computed — silently scoring from empty/garbage data would invalidate the entire layer-selection result with no visible error.

In [ ]:
%%writefile compute_mmd.py
import os
import sys
import torch
from collections import defaultdict
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

sys.path.insert(0, "openvla")
from prismatic.vla.datasets import RLDSDataset, RLDSBatchTransform
from prismatic.vla.action_tokenizer import ActionTokenizer
from prismatic.util.data_utils import PaddedCollatorForActionPrediction
from prismatic.models.backbones.llm.prompting import PurePromptBuilder


def rbf_kernel(x, y, bandwidth):
    dist = torch.cdist(x, y, p=2) ** 2
    return torch.exp(-dist / (2 * bandwidth ** 2))


def compute_mmd(x, y):
    combined = torch.cat([x, y], dim=0)
    pdist = torch.pdist(combined, p=2)
    bandwidth = torch.median(pdist) if pdist.numel() > 0 else torch.tensor(1.0)
    bandwidth = max(bandwidth.item(), 1e-4)

    xx = rbf_kernel(x, x, bandwidth)
    yy = rbf_kernel(y, y, bandwidth)
    xy = rbf_kernel(x, y, bandwidth)

    m, n = x.size(0), y.size(0)
    xx_sum = (torch.sum(xx) - torch.trace(xx)) / (m * (m - 1))
    yy_sum = (torch.sum(yy) - torch.trace(yy)) / (n * (n - 1))
    xy_sum = torch.sum(xy) / (m * n)
    return max(0.0, (xx_sum + yy_sum - 2 * xy_sum).item())


def main():
    num_samples = 64
    device = "cuda"

    print("Loading base model in 4-bit for MMD scoring...")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"
    )
    model = AutoModelForVision2Seq.from_pretrained(
        "openvla/openvla-7b",
        torch_dtype=torch.float16,
        quantization_config=quantization_config,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.eval()

    num_layers = len(model.language_model.model.layers)
    print(f"Confirmed {num_layers} LLM decoder layers at model.language_model.model.layers")

    activations = defaultdict(list)

    def get_hook(layer_idx):
        def hook(module, inp, output):
            hidden_states = output[0]
            activations[layer_idx].append(hidden_states.mean(dim=1).detach())
        return hook

    hooks = [model.language_model.model.layers[i].register_forward_hook(get_hook(i))
             for i in range(num_layers)]

    processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
    action_tokenizer = ActionTokenizer(processor.tokenizer)
    batch_transform = RLDSBatchTransform(
        action_tokenizer,
        processor.tokenizer,
        image_transform=processor.image_processor.apply_transform,
        prompt_builder_fn=PurePromptBuilder,
    )
    collator = PaddedCollatorForActionPrediction(
        processor.tokenizer.model_max_length, processor.tokenizer.pad_token_id, padding_side="right"
    )

    def collect_activations(dataset_name, label):
        print(f"Collecting activations for {label} ({dataset_name})...")
        dataset = RLDSDataset(
            "/kaggle/working/datasets/openvla/modified_libero_rlds",
            dataset_name,
            batch_transform,
            resize_resolution=tuple(model.config.image_sizes),
            shuffle_buffer_size=1000,
            image_aug=False,
        )
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, collate_fn=collator)

        activations.clear()
        collected, failures = 0, 0
        with torch.no_grad():
            for batch in dataloader:
                if collected >= num_samples:
                    break
                try:
                    model(
                        input_ids=batch["input_ids"].to(device),
                        attention_mask=batch["attention_mask"].to(device),
                        pixel_values=batch["pixel_values"].to(torch.float16).to(device),
                        labels=batch["labels"],
                    )
                    collected += 1
                    if collected % 16 == 0:
                        print(f"  ... {collected}/{num_samples}")
                except Exception as e:
                    failures += 1
                    print(f"  Batch failed ({failures} so far) — {type(e).__name__}: {e}")
                    continue

        if collected == 0:
            raise RuntimeError(
                f"\u274c Collected ZERO successful batches for {dataset_name} "
                f"({failures} failures). Cannot compute MMD from no data. "
                "Fix the underlying error above before proceeding."
            )

        for layer_idx in range(num_layers):
            assert len(activations[layer_idx]) > 0, (
                f"\u274c CRITICAL: zero activations captured for layer {layer_idx} "
                f"on {dataset_name}, despite {collected} successful batches. "
                "The forward hook may not be firing correctly."
            )

        print(f"✅ {label}: {collected} successful batches, {failures} failures")
        return {k: torch.cat(v, dim=0) for k, v in activations.items()}

    source_acts = collect_activations("libero_spatial_no_noops", "Source")
    target_acts = collect_activations("libero_object_no_noops", "Target")

    for h in hooks:
        h.remove()

    print("\n=== MMD Scores by Layer ===")
    scores = []
    for i in range(num_layers):
        mmd = compute_mmd(source_acts[i].float(), target_acts[i].float())
        scores.append((i, mmd))
        print(f"Layer {i:2d}: MMD = {mmd:.6f}")

    scores.sort(key=lambda x: x[1], reverse=True)
    with open("mmd_scores.txt", "w") as f:
        for idx, score in scores:
            f.write(f"{idx},{score}\n")

    print("\n✅ MMD scores saved to mmd_scores.txt (sorted highest to lowest)")


if __name__ == "__main__":
    main()


Run the scoring script:

In [ ]:
!PYTHONPATH={OPENVLA_DIR} python compute_mmd.py

## 9. Select layers to match the baseline's REAL parameter budget, then VERIFY the resulting `target_modules` regex directly

This is the fix for the gap that mattered most: rather than writing a `target_modules` regex into the training script and hoping it wraps the right modules, this cell **actually instantiates the model** (using `init_empty_weights` — no real weights or GPU needed, same technique already proven earlier in this project), runs `get_peft_model` against it with the exact candidate regex, and inspects the real resulting module list and parameter count. If it doesn't match the target budget, it is NOT silently accepted — the cell raises.

Budget target: the baseline's own real, printed trainable-parameter count (**110,828,288**) from its successful run — not a theoretical estimate. To hit this exactly, the selective config must also include `lm_head` and the two vision-backbone Conv2d patch-embed modules (`vision_backbone.featurizer.patch_embed.proj`, `vision_backbone.fused_featurizer.patch_embed.proj`) identically to the baseline — PEFT's `"all-linear"` wraps these too, alongside the 32 LLM layers and the rest of the vision backbone/projector, and the baseline's real total only reconciles if these are included on both sides.

In [ ]:
import re
from transformers import AutoConfig, AutoModelForVision2Seq
from accelerate import init_empty_weights
from peft import get_peft_model, LoraConfig

BASELINE_REAL_PARAM_COUNT = 110_828_288  # from the baseline's actual successful run

with open("mmd_scores.txt") as f:
    scores = [line.strip().split(",") for line in f if line.strip()]
scores = [(int(idx), float(s)) for idx, s in scores]  # already sorted highest MMD first

# Fixed, non-LLM-layer part of the baseline's target set (kept IDENTICAL between
# conditions on purpose, so the only real experimental difference is LLM-layer choice)
FIXED_TAIL = r"(vision_backbone|projector)\..*(fc1|fc2|fc3|proj|q|kv|qkv)|language_model\.lm_head"

def build_regex(selected_layer_indices):
    layer_pattern = "|".join(str(i) for i in sorted(selected_layer_indices))
    return (
        rf"^(language_model\.model\.layers\.({layer_pattern})\."
        r"(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))"
        rf"|{FIXED_TAIL})$"
    )

def real_param_count_for_regex(regex_str):
    """Actually instantiate the model and get_peft_model it — real evidence, not a guess."""
    config = AutoConfig.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
    with init_empty_weights():
        model = AutoModelForVision2Seq.from_config(config, trust_remote_code=True)
    lora_config = LoraConfig(
        r=32, lora_alpha=min(32, 16), lora_dropout=0.0,
        target_modules=regex_str, init_lora_weights="gaussian",
    )
    model = get_peft_model(model, lora_config)
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    wrapped = sorted({n for n, m in model.named_modules() if "lora_A" in dir(m)})
    return total, wrapped

# Try increasing numbers of top-MMD layers until we match or pass the baseline budget
best = None
for k in range(1, 33):
    selected = [idx for idx, _ in scores[:k]]
    regex_str = build_regex(selected)
    total, wrapped = real_param_count_for_regex(regex_str)
    diff = total - BASELINE_REAL_PARAM_COUNT
    print(f"K={k:2d} layers: {total:,} params (baseline diff: {diff:+,})")
    if best is None or abs(diff) < abs(best[1] - BASELINE_REAL_PARAM_COUNT):
        best = (k, total, selected, regex_str, wrapped)
    if total >= BASELINE_REAL_PARAM_COUNT:
        break

K, achieved_total, SELECTED_LAYERS, TARGET_MODULES_REGEX, wrapped_modules = best

print(f"\n✅ SELECTED K={K} layers: {SELECTED_LAYERS}")
print(f"✅ Achieved parameter count: {achieved_total:,} (baseline: {BASELINE_REAL_PARAM_COUNT:,}, "
      f"diff: {achieved_total - BASELINE_REAL_PARAM_COUNT:+,})")
print(f"\nFinal target_modules regex:\n{TARGET_MODULES_REGEX}")
print(f"\nTotal wrapped module count: {len(wrapped_modules)}")
print("Sample of wrapped modules (first 10):")
for m in wrapped_modules[:10]:
    print(f"  {m}")

# Hard checks — do not proceed silently if these fail
assert len(wrapped_modules) > 0, "❌ Regex matched ZERO modules — LoRA would attach nothing."
assert any("lm_head" in m for m in wrapped_modules), (
    "❌ lm_head was not wrapped — parameter budget will not match the baseline."
)
assert any(f"layers.{SELECTED_LAYERS[0]}." in m for m in wrapped_modules), (
    "❌ The selected LLM layers do not appear to be wrapped — regex may be malformed."
)
# Allow a small tolerance since exact equality across arbitrary K may not be achievable
tolerance = 0.02  # 2%
rel_diff = abs(achieved_total - BASELINE_REAL_PARAM_COUNT) / BASELINE_REAL_PARAM_COUNT
assert rel_diff <= tolerance, (
    f"❌ Achieved parameter count differs from baseline by {rel_diff:.1%}, "
    f"exceeding the {tolerance:.0%} tolerance. Selection is not budget-matched."
)
print(f"\n✅ VERIFIED: within {rel_diff:.2%} of baseline budget, regex confirmed to wrap "
      "the intended modules by direct inspection.")

with open("selected_layers.txt", "w") as f:
    f.write(",".join(map(str, SELECTED_LAYERS)))
with open("target_modules_regex.txt", "w") as f:
    f.write(TARGET_MODULES_REGEX)


## 10. Generate `finetune_mmd_selective.py`

Copies the ORIGINAL `finetune.py` (not the already-patched one from section 4, to avoid double-patching) and applies: the same fp16 patch (all occurrences), the same proven DDP+accelerate fixes are already applied globally in section 7 above (they patch the installed library and `finetune.py` in place, so this new script inherits them), the same `shuffle_buffer_size` reduction, `WANDB_MODE=disabled`, and — the only real difference from the baseline — `target_modules` swapped from `"all-linear"` to the VERIFIED regex from section 9.

In [ ]:
with open("selected_layers.txt") as f:
    print(f"Using verified layer selection: {f.read()}")
with open("target_modules_regex.txt") as f:
    verified_regex = f.read()

finetune_file = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune.py")
with open(finetune_file, "r") as f:
    content = f.read()

old_target = 'target_modules="all-linear"'
assert old_target in content, (
    "❌ Could not find target_modules=\"all-linear\" in finetune.py — "
    "has this file already been modified in a way that changes this line?"
)
new_target = f'target_modules=r"{verified_regex}"'
content = content.replace(old_target, new_target)

selective_script = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune_mmd_selective.py")
with open(selective_script, "w") as f:
    f.write(content)

# Sanity check: the new script differs from the original ONLY in target_modules
with open(finetune_file, "r") as f:
    baseline_ft_content = f.read()
diff_check = baseline_ft_content.replace(old_target, new_target)
assert diff_check == content, (
    "❌ finetune_mmd_selective.py diverges from finetune.py in more than just "
    "target_modules — something unexpected happened during generation."
)
print(f"✅ Created {selective_script}")
print("✅ Confirmed: differs from the (already-patched, proven-working) finetune.py "
      "in target_modules ONLY.")


## 11. Launch the MMD-selective training run (100 steps — smoke test)

Same launch mechanism as the proven baseline (`torch.distributed.run`, not `accelerate launch`), `batch_size=1` (the only proven-safe value), same `shuffle_buffer_size` fix inherited from the globally-patched `finetune.py`. Run/checkpoint directory name distinct from the baseline's, so nothing overwrites it.

In [ ]:
RUN_NAME = "mmd_selective_lora_run"
CHECKPOINT_DIR = os.path.join(WORKDIR, "openvla_checkpoints", RUN_NAME)
MAX_STEPS = 100   # smoke test — scale this the same way the baseline was scaled, once this passes
SAVE_STEPS = 100

os.environ["WANDB_MODE"] = "disabled"
if os.path.exists(CHECKPOINT_DIR):
    print(f"⚠️ {CHECKPOINT_DIR} exists from a previous attempt — removing it to start clean.")
    shutil.rmtree(CHECKPOINT_DIR)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
adapter_tmp_dir = os.path.join(CHECKPOINT_DIR, "tmp")

selective_script = os.path.join(OPENVLA_DIR, "vla-scripts", "finetune_mmd_selective.py")

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", "--nnodes=1", "--nproc-per-node=1",
    selective_script,
    "--vla_path", "openvla/openvla-7b",
    "--data_root_dir", DATA_DIR,
    "--dataset_name", "libero_object_no_noops",
    "--run_root_dir", CHECKPOINT_DIR,
    "--adapter_tmp_dir", adapter_tmp_dir,
    "--use_lora", "True",
    "--use_quantization", "True",
    "--lora_rank", "32",
    "--batch_size", "1",
    "--grad_accumulation_steps", "1",
    "--learning_rate", "5e-4",
    "--image_aug", "False",
    "--wandb_project", "",
    "--wandb_entity", "",
    "--save_steps", str(SAVE_STEPS),
    "--max_steps", str(MAX_STEPS),
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["WANDB_MODE"] = "disabled"

print("\n" + "=" * 70)
print(f"STARTING MMD-SELECTIVE RUN: {RUN_NAME}")
print("=" * 70)
print(f"Selected layers: {open('selected_layers.txt').read()}")
print(f"Command:\n" + " ".join(cmd))
print("=" * 70 + "\n")

log_path = os.path.join(CHECKPOINT_DIR, "training.log")
with open(log_path, "w") as log_file:
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, cwd=OPENVLA_DIR, env=env,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    process.wait()

if process.returncode != 0:
    print(f"❌ Training failed with exit code {process.returncode}")
    raise RuntimeError(f"MMD-selective run failed with exit code {process.returncode}")
print(f"✅ Training process exited successfully! ({MAX_STEPS} steps)")


## 12. End-to-end verification

Same checks as the baseline's verification cell.

In [ ]:
import glob
import json

with open(os.path.join(CHECKPOINT_DIR, "training.log"), "r") as f:
    logs = f.read()

step_marker_found = (
    f"Step {MAX_STEPS}" in logs or f"step {MAX_STEPS}" in logs
    or f"{MAX_STEPS}/{MAX_STEPS}" in logs or f"Max step {MAX_STEPS} reached" in logs
)
assert step_marker_found, f"❌ Training did not demonstrably reach step {MAX_STEPS}."
assert "trainable params" in logs, "❌ Could not verify trainable LoRA parameters."
print(f"✅ Log verification: training reached step {MAX_STEPS}, PEFT attached.")

adapters = glob.glob(os.path.join(CHECKPOINT_DIR, "**", "adapter_model.safetensors"), recursive=True)
configs = glob.glob(os.path.join(CHECKPOINT_DIR, "**", "adapter_config.json"), recursive=True)
assert adapters, "❌ No LoRA adapter was saved."
assert configs, "❌ No adapter_config.json was saved."

with open(configs[0]) as f:
    cfg = json.load(f)
print(f"✅ Found adapter at: {adapters[0]}")
print(f"LoRA rank (r): {cfg.get('r')}")

from peft import PeftConfig
peft_cfg = PeftConfig.from_pretrained(os.path.dirname(configs[0]))
assert peft_cfg.r == 32, "❌ Reloaded PEFT config rank does not match."
print("✅ PEFT adapter configuration is valid and loadable.")

print(f"\n🚀 MMD-SELECTIVE SMOKE TEST PASS: {RUN_NAME}, {MAX_STEPS} steps, "
      f"{K} layers selected, checkpoint at {CHECKPOINT_DIR}")
print("Next: scale MAX_STEPS to match the baseline's 2000-step real run, the same way "
      "the baseline itself was scaled from its own smoke test.")
